# Molecular property prediction #

In [4]:
from sentence_transformers import SentenceTransformer
import numpy as np
import json
from sklearn.neighbors import KNeighborsClassifier
import contriever.src.contriever
import torch
from tqdm import tqdm
import pandas as pd
import os
import random
# 设置随机种子
random.seed(42)
np.random.seed(42)

print("✓ 正在初始化环境...")

✓ 正在初始化环境...


# 1. Main function #

In [5]:
def embed_queries_internal(queries, model, tokenizer):
    device = 'cuda:0'
    model.eval()
    embeddings, batch_question = [], []
    with torch.no_grad():
        for k, q in enumerate(queries):
            batch_question.append(q)
            encoded_batch = tokenizer.batch_encode_plus(
                batch_question, return_tensors="pt", max_length=512, padding=True, truncation=True,
            )
            encoded_batch = {k: v.to(device) for k, v in encoded_batch.items()}
            output = model(**encoded_batch)
            embeddings.append(output.cpu())
            batch_question = []
    embeddings = torch.cat(embeddings, dim=0).numpy()
    return embeddings

def get_train_embedding(fname, key_name, normalize_embeddings, model_name):
    with open(fname) as f:
        data = json.load(f)
    data = data[key_name]
    train_embedding = {}
    print("正在编码训练数据...")
    for key, values in tqdm(data.items()):
        if model_name == 'OpenScholar_Reranker':
            train_embedding[key] = model.encode(values, normalize_embeddings=normalize_embeddings)
        else:
            train_embedding[key] = embed_queries_internal(values, query_encoder, query_tokenizer)
    return train_embedding


# 2. Loading the model and data #

In [9]:
model_name = 'contriever'
print(f"\n--- Step 1: Loading Model ({model_name}) ---")

if model_name == 'OpenScholar_Reranker':
    model = SentenceTransformer('/data/oyyw/llm/OpenScholar_Reranker')
    model = model.to('cuda:0')
    normalize_embeddings = True
    model.eval()
else:
    query_encoder, query_tokenizer, _ = contriever.src.contriever.load_retriever('akariasai/pes2o_contriever')
    query_encoder = query_encoder.to('cuda:0')
    normalize_embeddings = True
    query_encoder.eval()

# 加载训练数据
fname = 'merged.json'
if not os.path.exists(fname):
    print(f"错误：找不到训练数据 {fname}")
    exit(1)
train_embedding = get_train_embedding(fname, 'all_description', normalize_embeddings, model_name)

# 加载测试数据 (Final 90)
excel_path = 'Molecule_Information_90.xlsx'
if not os.path.exists(excel_path):
    print(f"错误：找不到文件 {excel_path}")
    exit(1)

print(f"\n--- Step 2: Loading Data from {excel_path} ---")
df = pd.read_excel(excel_path)
df.columns = df.columns.str.strip()
if 'Final_ID' not in df.columns:
    df['Final_ID'] = range(1, len(df) + 1)

materials = df['Description'].astype(str).tolist()
ids = df['Final_ID'].tolist()
names = df['Abbreviation'].tolist()

print("正在编码测试数据...")
test_embedding = embed_queries_internal(materials, query_encoder, query_tokenizer)


--- Step 1: Loading Model (contriever) ---
正在编码训练数据...


100%|██████████| 10/10 [00:02<00:00,  3.66it/s]



--- Step 2: Loading Data from Molecule_Information_90.xlsx ---
正在编码测试数据...



# 3. Prediction (Stiffness & Ion Migration) #

In [7]:
print("\n--- Step 3: Prediction ---")

target_props = {
    'Stiffness': 'stiffness',
    'Ion_Migration': 'Ion migration activation energy of the crystal'
}

results = {}

for short_name, prop_full in target_props.items():
    print(f"  预测：{short_name}...")
    property_key_high = f'high {prop_full}'
    property_key_low = f'low {prop_full}'

    if property_key_high not in train_embedding:
        print(f"  警告：训练数据中找不到 '{property_key_high}'，跳过。")
        continue

    x_train = np.concatenate([train_embedding[property_key_high], train_embedding[property_key_low]], axis=0)
    y_train = np.array(len(train_embedding[property_key_high]) * [1] + len(train_embedding[property_key_low]) * [0])

    k_values = list(range(20, 100, 5))
    prob_matrix_temp = []

    for n_neighbors in k_values:
        curr_k = min(n_neighbors, len(x_train))
        knn = KNeighborsClassifier(n_neighbors=curr_k)
        knn.fit(x_train, y_train)
        y_pred = knn.predict_proba(test_embedding)
        prob_matrix_temp.append(y_pred[:, 1])

    results[short_name] = np.mean(prob_matrix_temp, axis=0)


--- Step 3: Prediction ---
  预测：Stiffness...
  预测：Ion_Migration...



# 4. Output the predicted results #

In [8]:
print("\n--- Step 4: Saving Prediction Results ---")

df_result = pd.DataFrame({
    'Final_ID'         : ids,
    'Abbreviation'     : names,
    'Stiffness_Prob'   : results.get('Stiffness',    [None] * len(ids)),
    'Ion_Migration_Prob': results.get('Ion_Migration', [None] * len(ids))
})

output_path = 'Prediction_Results.xlsx'
df_result.to_excel(output_path, index=False)

print(f"✓ 预测结果已保存至：{output_path}")


--- Step 4: Saving Prediction Results ---
✓ 预测结果已保存至：Prediction_Results.xlsx



# 5. Visualization Module #

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import datetime

print("\n--- Step 5: Visualization ---")

timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
output_dir = f'output_{timestamp}'
os.makedirs(output_dir, exist_ok=True)

sns.set_theme(style="whitegrid")

# ── 图1: 双属性条形图 (所有分子) ──────────────────────────────
print("  [绘图] 双属性条形图...")

df_melt = df_result.melt(
    id_vars=['Final_ID', 'Abbreviation'],
    value_vars=['Stiffness_Prob', 'Ion_Migration_Prob'],
    var_name='Property',
    value_name='Probability'
)
df_melt['Property'] = df_melt['Property'].replace({
    'Stiffness_Prob'    : 'Stiffness',
    'Ion_Migration_Prob': 'Ion Migration'
})
df_melt['Label'] = df_melt.apply(lambda x: f"{x['Final_ID']}. {x['Abbreviation']}", axis=1)

plt.figure(figsize=(12, max(8, len(ids) * 0.28)))
sns.barplot(data=df_melt, y='Label', x='Probability', hue='Property', palette='coolwarm')
plt.title('All Molecules: Stiffness & Ion Migration Probability', fontsize=14)
plt.xlabel('Probability')
plt.ylabel('Molecule')
plt.xlim(0, 1.1)
plt.legend(loc='lower right')
plt.tight_layout()
chart1_path = f"{output_dir}/Chart1_All_Barplot.png"
plt.savefig(chart1_path, dpi=200)
plt.close()
print(f"  [保存] {chart1_path}")

print(f"\n✓ 全部完成！结果文件夹：{output_dir}")
